# 05履约风险优先级与运营干预方案
本阶段基于03销售基本盘和04履约评分分析的结果，进一步识别高业务价值且高履约风险的订单和地区。前序分析已经发现，延迟订单的平均评分明显低于准时或提前订单，且1星差评率显著更高。因此，本阶段将重点分析“哪些订单或地区应该优先被运营干预”。

本阶段主要完成三项任务：第一，结合GMV、订单量、延迟率和差评率识别重点地区；第二，基于延迟天数构建订单风险分层；第三，提出延迟订单主动干预和A/B测试设计方案。

## 1 读取数据

In [19]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data_clean")

order_base = pd.read_csv(DATA_DIR / "order_base.csv")

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in date_cols:
    order_base[col] = pd.to_datetime(order_base[col], errors="coerce")

risk_base = order_base[
    (order_base["order_status"] == "delivered")
    & order_base["payment_total"].notna()
    & order_base["price_total"].notna()
    & order_base["freight_total"].notna()
    & order_base["delivery_days"].notna()
    & order_base["delay_days"].notna()
    & order_base["review_score"].notna()
].copy()

risk_base["is_late"] = risk_base["delay_days"] > 0
risk_base["is_one_star"] = risk_base["review_score"] == 1
risk_base["is_low_score"] = risk_base["review_score"] <= 2

print("risk_base订单数:", risk_base["order_id"].nunique())
print("order_id是否唯一:", risk_base["order_id"].is_unique)

risk_base订单数: 95823
order_id是否唯一: True


**观察与总结**

本阶段使用已送达、金额字段完整、履约字段完整且存在评分的订单作为分析样本。这样可以同时支持GMV、延迟率、差评率和风险分层分析。该样本是03销售分析样本和04履约评分分析样本的交集，因此更适合用于后续业务优先级判断。

## 2 地区风险优先级

### 2.1 地区数据概览

In [20]:
state_risk = (
    risk_base
    .groupby("customer_state", as_index=False)
    .agg(
        orders=("order_id", "nunique"),
        gmv=("payment_total", "sum"),
        avg_order_value=("payment_total", "mean"),
        avg_delivery_days=("delivery_days", "mean"),
        avg_delay_days=("delay_days", "mean"),
        late_rate=("is_late", "mean"),
        avg_review_score=("review_score", "mean"),
        one_star_rate=("is_one_star", "mean"),
        low_score_rate=("is_low_score", "mean"),
    )
)

state_risk["gmv_share"] = state_risk["gmv"] / state_risk["gmv"].sum()

state_risk[
    [
        "gmv",
        "avg_order_value",
        "avg_delivery_days",
        "avg_delay_days",
        "avg_review_score",
    ]
] = state_risk[
    [
        "gmv",
        "avg_order_value",
        "avg_delivery_days",
        "avg_delay_days",
        "avg_review_score",
    ]
].round(2)

state_risk[
    [
        "late_rate",
        "one_star_rate",
        "low_score_rate",
        "gmv_share",
    ]
] = state_risk[
    [
        "late_rate",
        "one_star_rate",
        "low_score_rate",
        "gmv_share",
    ]
].round(4)

display(
    state_risk
    .sort_values("gmv", ascending=False)
    .head(10)
)

,customer_state,orders,gmv,avg_order_value,avg_delivery_days,avg_delay_days,late_rate,avg_review_score,one_star_rate,low_score_rate,gmv_share
25,SP,40265,5736983.42,142.48,8.75,-10.40,0.0583,4.25,0.0787,0.1064,0.3751
18,RJ,12211,2026600.23,165.97,15.24,-11.11,0.1328,3.97,0.1471,0.1829,0.1325
10,MG,11285,1804521.41,159.90,11.98,-12.56,0.0550,4.19,0.0891,0.1171,0.1180
22,RS,5326,858654.84,161.22,15.28,-13.22,0.0708,4.19,0.0894,0.1189,0.0561
17,PR,4900,779570.14,159.10,11.97,-12.65,0.0492,4.24,0.0792,0.1084,0.0510
4,BA,3229,587085.36,181.82,19.25,-10.20,0.1369,3.93,0.1316,0.1703,0.0384
23,SC,3519,586512.00,166.67,14.88,-10.87,0.0963,4.13,0.0980,0.1302,0.0384
6,DF,2070,344789.44,166.56,12.96,-11.33,0.0710,4.13,0.1019,0.1304,0.0225
8,GO,1946,330127.14,169.64,15.56,-11.54,0.0791,4.10,0.0976,0.1310,0.0216
15,PE,1579,306980.68,194.41,18.37,-12.67,0.1064,4.08,0.1159,0.1482,0.0201


### 2.2 地区履约治理优先级排序

In [21]:
min_orders = 500

state_priority = state_risk[state_risk["orders"] >= min_orders].copy()

state_priority["gmv_rank"] = state_priority["gmv"].rank(
    ascending=False,
    method="min",
)

state_priority["late_rate_rank"] = state_priority["late_rate"].rank(
    ascending=False,
    method="min",
)

state_priority["one_star_rate_rank"] = state_priority["one_star_rate"].rank(
    ascending=False,
    method="min",
)

state_priority["priority_score"] = (
    state_priority["gmv_rank"]
    + state_priority["late_rate_rank"]
    + state_priority["one_star_rate_rank"]
)

state_priority = state_priority.sort_values("priority_score")

display(
    state_priority[
        [
            "customer_state",
            "orders",
            "gmv",
            "gmv_rank",
            "late_rate",
            "late_rate_rank",
            "one_star_rate",
            "one_star_rate_rank",
            "priority_score",
        ]
    ].head(10)
)

,customer_state,orders,gmv,gmv_rank,late_rate,late_rate_rank,one_star_rate,one_star_rate_rank,priority_score
18,RJ,12211,2026600.23,2.0,0.1328,4.0,0.1471,2.0,8.0
4,BA,3229,587085.36,6.0,0.1369,3.0,0.1316,5.0,14.0
9,MA,712,146912.41,15.0,0.1924,1.0,0.1573,1.0,17.0
5,CE,1273,265283.55,12.0,0.1524,2.0,0.1351,4.0,18.0
13,PA,933,207807.41,13.0,0.1190,5.0,0.1436,3.0,21.0
7,ES,1969,305437.00,11.0,0.1188,6.0,0.1097,7.0,24.0
15,PE,1579,306980.68,10.0,0.1064,9.0,0.1159,6.0,25.0
23,SC,3519,586512.00,7.0,0.0963,10.0,0.0980,11.0,28.0
6,DF,2070,344789.44,8.0,0.0710,12.0,0.1019,9.0,29.0
22,RS,5326,858654.84,4.0,0.0708,13.0,0.0894,14.0,31.0


**观察与总结**

本节基于地区维度构建了一个简单的履约治理优先级排序。排序同时考虑GMV、延迟率和1星差评率三个因素：GMV越高，说明该地区业务影响越大；延迟率越高，说明履约问题越突出；1星差评率越高，说明客户负面体验风险越高。

从排序结果看，RJ位列优先级第一。该地区GMV排名第2，同时延迟率和1星差评率也处于较高水平，说明其履约问题不仅影响客户体验，也可能影响较大的业务规模。BA同样具有较高的GMV和延迟风险，适合作为第二梯队重点治理地区。

MA和CE虽然GMV规模不如SP、RJ等核心市场，但延迟率和1星差评率较高，说明这些地区可能存在更突出的履约稳定性问题。对于这类地区，可以优先进行物流链路排查，而不是简单按照GMV大小排序。

本节主要价值在于将销售规模、履约风险和客户反馈合并到同一张表中，帮助运营侧快速识别优先治理地区。

## 3 订单风险分层

In [22]:
risk_bins = [-np.inf, 0, 3, 7, np.inf]

risk_labels = [
    "low_risk_on_time_or_early",
    "medium_risk_late_0_3d",
    "high_risk_late_3_7d",
    "critical_risk_late_7d_plus",
]

risk_base["delivery_risk_level"] = pd.cut(
    risk_base["delay_days"],
    bins=risk_bins,
    labels=risk_labels,
)

risk_level_summary = (
    risk_base
    .groupby("delivery_risk_level", observed=True, as_index=False)
    .agg(
        orders=("order_id", "nunique"),
        gmv=("payment_total", "sum"),
        avg_delay_days=("delay_days", "mean"),
        avg_review_score=("review_score", "mean"),
        one_star_rate=("is_one_star", "mean"),
        low_score_rate=("is_low_score", "mean"),
    )
)

risk_level_summary["order_share"] = (
    risk_level_summary["orders"] / risk_level_summary["orders"].sum()
)

risk_level_summary["gmv_share"] = (
    risk_level_summary["gmv"] / risk_level_summary["gmv"].sum()
)

risk_level_summary[
    [
        "gmv",
        "avg_delay_days",
        "avg_review_score",
    ]
] = risk_level_summary[
    [
        "gmv",
        "avg_delay_days",
        "avg_review_score",
    ]
].round(2)

risk_level_summary[
    [
        "one_star_rate",
        "low_score_rate",
        "order_share",
        "gmv_share",
    ]
] = risk_level_summary[
    [
        "one_star_rate",
        "low_score_rate",
        "order_share",
        "gmv_share",
    ]
].round(4)

display(risk_level_summary)

,delivery_risk_level,orders,gmv,avg_delay_days,avg_review_score,one_star_rate,low_score_rate,order_share,gmv_share
0,low_risk_on_time_or_early,88169,13981464.89,-13.01,4.29,0.0655,0.0919,0.9201,0.9143
1,medium_risk_late_0_3d,2634,414033.39,1.41,3.76,0.1378,0.1917,0.0275,0.0271
2,high_risk_late_3_7d,1773,307480.05,5.12,2.32,0.5302,0.6125,0.0185,0.0201
3,critical_risk_late_7d_plus,3247,589673.84,18.34,1.73,0.6865,0.7835,0.0339,0.0386


**观察与总结**

本节将订单按照delay_days划分为低风险、中风险、高风险和严重风险。该分层不是为了精确预测评分，而是为了将04中发现的延迟风险转化为可执行的运营规则。

低风险订单为准时或提前送达订单，通常评分表现较好；中风险订单为延迟0到3天，适合采用提醒类干预；高风险和严重风险订单差评风险更高，适合优先纳入客服跟进或补偿策略。通过统计各风险层的订单量、GMV和差评率，可以判断不同干预策略大约会覆盖多少订单和业务规模。

## 4 干预策略建议

In [23]:
action_plan = pd.DataFrame(
    {
        "risk_level": [
            "low_risk_on_time_or_early",
            "medium_risk_late_0_3d",
            "high_risk_late_3_7d",
            "critical_risk_late_7d_plus",
        ],
        "trigger_rule": [
            "delay_days <= 0",
            "0 < delay_days <= 3",
            "3 < delay_days <= 7",
            "delay_days > 7",
        ],
        "suggested_action": [
            "不额外干预，维持正常服务",
            "发送延迟提醒和物流状态说明",
            "主动客服跟进，并提供小额优惠券或运费补偿",
            "优先客服介入，提供更强补偿并追踪后续评价",
        ],
        "business_goal": [
            "保持正常满意度",
            "降低不确定性和用户焦虑",
            "降低1星差评和低分评价风险",
            "挽回高风险订单体验，减少极端差评",
        ],
    }
)

display(action_plan)

,risk_level,trigger_rule,suggested_action,business_goal
0,low_risk_on_time_or_early,delay_days <= 0,不额外干预，维持正常服务,保持正常满意度
1,medium_risk_late_0_3d,0 < delay_days <= 3,发送延迟提醒和物流状态说明,降低不确定性和用户焦虑
2,high_risk_late_3_7d,3 < delay_days <= 7,主动客服跟进，并提供小额优惠券或运费补偿,降低1星差评和低分评价风险
3,critical_risk_late_7d_plus,delay_days > 7,优先客服介入，提供更强补偿并追踪后续评价,挽回高风险订单体验，减少极端差评


## 5 干预覆盖规模评估
本节将风险分层结果与干预动作进行匹配，并进一步统计高风险和严重风险订单的覆盖规模，用于评估主动干预策略的大致影响范围。

### 5.1 风险层级与干预动作匹配

In [24]:
intervention_summary = risk_level_summary.copy()

intervention_summary["risk_level"] = intervention_summary[
    "delivery_risk_level"
].astype(str)

intervention_summary = intervention_summary.merge(
    action_plan,
    on="risk_level",
    how="left",
)

display(
    intervention_summary[
        [
            "risk_level",
            "orders",
            "order_share",
            "gmv",
            "gmv_share",
            "avg_review_score",
            "one_star_rate",
            "low_score_rate",
            "suggested_action",
        ]
    ]
)

,risk_level,orders,order_share,gmv,gmv_share,avg_review_score,one_star_rate,low_score_rate,suggested_action
0,low_risk_on_time_or_early,88169,0.9201,13981464.89,0.9143,4.29,0.0655,0.0919,不额外干预，维持正常服务
1,medium_risk_late_0_3d,2634,0.0275,414033.39,0.0271,3.76,0.1378,0.1917,发送延迟提醒和物流状态说明
2,high_risk_late_3_7d,1773,0.0185,307480.05,0.0201,2.32,0.5302,0.6125,主动客服跟进，并提供小额优惠券或运费补偿
3,critical_risk_late_7d_plus,3247,0.0339,589673.84,0.0386,1.73,0.6865,0.7835,优先客服介入，提供更强补偿并追踪后续评价


### 5.2 高风险订单覆盖规模评估
单独估算真正需要优先干预的高风险和严重风险订单规模

In [25]:
high_risk_orders = risk_level_summary.loc[
    risk_level_summary["delivery_risk_level"].isin(
        [
            "high_risk_late_3_7d",
            "critical_risk_late_7d_plus",
        ]
    ),
    "orders",
].sum()

high_risk_gmv = risk_level_summary.loc[
    risk_level_summary["delivery_risk_level"].isin(
        [
            "high_risk_late_3_7d",
            "critical_risk_late_7d_plus",
        ]
    ),
    "gmv",
].sum()

high_risk_order_share = high_risk_orders / risk_level_summary["orders"].sum()
high_risk_gmv_share = high_risk_gmv / risk_level_summary["gmv"].sum()

high_risk_summary = (
    pd.Series(
        {
            "high_and_critical_orders": high_risk_orders,
            "high_and_critical_order_share": high_risk_order_share,
            "high_and_critical_gmv": high_risk_gmv,
            "high_and_critical_gmv_share": high_risk_gmv_share,
        }
    )
    .round(4)
    .to_frame("value")
)

display(high_risk_summary)

,value
high_and_critical_orders,5020.0000
high_and_critical_order_share,0.0524
high_and_critical_gmv,897153.8900
high_and_critical_gmv_share,0.0587


**观察与总结**

从风险分层结果看，延迟3天以上的高风险和严重风险订单合计约5020单，占样本订单约5.24%，对应GMV约89.72万，占样本GMV约5.87%。虽然这部分订单占比不高，但其1星差评率明显高于准时或轻微延迟订单，是最值得优先干预的订单群体。

其中，延迟3到7天订单的1星差评率已超过50%，延迟7天以上订单的1星差评率接近70%。这说明延迟超过3天后，客户极端负面评价风险明显放大。因此，平台可以将delay_days>3作为主动预警阈值，将delay_days>7作为强干预阈值。

## 6 A/B测试设计方案

In [26]:
ab_test_design = pd.DataFrame(
    {
        "module": [
            "实验目的",
            "实验对象",
            "对照组A",
            "实验组B",
            "核心指标",
            "辅助指标",
            "风险控制",
            "判断标准",
        ],
        "design": [
            "验证主动通知或补偿是否能降低延迟订单差评率",
            "预计或已经延迟超过3天、尚未完成评价的已支付订单",
            "维持现有通知和售后流程",
            "主动发送延迟说明，并提供客服入口或小额补偿",
            "1星差评率、1-2星低分率、平均评分",
            "客服投诉率、后续复购率、补偿成本",
            "随机分组，并保证地区、订单金额、延迟天数分布相近",
            "若实验组差评率下降且补偿成本可控，则考虑上线策略",
        ],
    }
)

display(ab_test_design)

,module,design
0,实验目的,验证主动通知或补偿是否能降低延迟订单差评率
1,实验对象,预计或已经延迟超过3天、尚未完成评价的已支付订单
2,对照组A,维持现有通知和售后流程
3,实验组B,主动发送延迟说明，并提供客服入口或小额补偿
4,核心指标,1星差评率、1-2星低分率、平均评分
5,辅助指标,客服投诉率、后续复购率、补偿成本
6,风险控制,随机分组，并保证地区、订单金额、延迟天数分布相近
7,判断标准,若实验组差评率下降且补偿成本可控，则考虑上线策略


**观察与总结**

本节基于前文风险分层结果，提出针对延迟超过3天订单的主动干预实验方案。该方案将延迟订单随机分为对照组和实验组，对照组维持现有流程，实验组主动发送延迟说明并提供客服入口或小额补偿。

需要注意的是，当前Olist数据是历史观察数据，只能说明履约延迟与差评风险之间存在相关关系，不能直接验证干预策略的因果效果。因此，A/B测试设计的作用是为后续真实业务实验提供方案，而不是声称本项目已经完成实验验证。

## 总结

本阶段将前序销售分析和履约评分分析转化为运营优先级判断，重点回答“哪些地区和订单应该优先被干预”。

从地区维度看，RJ、BA等地区同时具备较高GMV和较高履约风险，适合优先纳入治理范围。MA、CE等地区虽然GMV规模相对较小，但延迟率和1星差评率较高，说明可能存在更突出的履约稳定性问题，适合进行专项排查。

从订单风险分层看，准时或提前订单占主体，评分表现较好；延迟0到3天订单属于中风险，可以通过提醒和物流状态说明降低用户不确定性；延迟3到7天和7天以上订单虽然占比不高，但1星差评率明显上升，是最值得优先干预的订单群体。

从干预覆盖规模看，延迟3天以上的高风险和严重风险订单合计约5020单，占样本订单约5.24%，对应GMV约89.72万，占样本GMV约5.87%。这部分订单规模不算最大，但差评风险高，适合作为主动通知、客服跟进或补偿策略的优先试点对象。

最后，本阶段提出了延迟订单主动干预的A/B测试设计方案。需要强调的是，当前项目只能基于历史数据识别风险和提出策略，不能直接证明干预方案有效。若要验证主动通知或补偿能否降低差评率，需要在真实业务中进行随机实验。